# **Import**

In [10]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
from data_loader import Dataset, standardize
from sklearn.linear_model import Lasso, Ridge, ElasticNet, LogisticRegression
from sklearn.metrics import mean_squared_error, accuracy_score

# **Build From Scratch**

In [11]:
def get_reg(reg_l1, reg_l2, weights):

    reg = 0

    if reg_l1 and reg_l2:
        reg = reg_l1 * np.sign(weights) + reg_l2 * weights

    elif reg_l1:
        reg = reg_l1 * np.sign(weights)

    elif reg_l2:
        reg = reg_l2 * weights

    return reg

In [12]:
class CustomLinearRegression():

    def __init__(self, reg_l1=None, reg_l2=None, learning_rate=0.001, num_iterations=10000):

        self.learning_rate = learning_rate
        self.num_iterations = num_iterations

        self.reg_l1 = reg_l1
        self.reg_l2 = reg_l2

    def fit(self, X, y):

        n, m = X.shape

        self.weights = np.random.randn(m)
        self.bias = np.random.randn()

        for _ in range(self.num_iterations):

            y_pred = self.predict(X)

            reg = get_reg(self.reg_l1, self.reg_l2, self.weights)

            dw = (2 / n) * np.dot(X.T, (y_pred - y)) + reg
            db = (2 / n) * np.sum(y_pred - y)

            self.weights -= self.learning_rate * dw
            self.bias -= self.learning_rate * db

    def predict(self, X):
        return np.dot(X, self.weights) + self.bias


In [13]:
class CustomLogisticRegression():

    def __init__(self, reg_l1=None, reg_l2=None,learning_rate=0.001, num_iterations=10000):

        self.learning_rate = learning_rate
        self.num_iterations = num_iterations

        self.reg_l1 = reg_l1
        self.reg_l2 = reg_l2


    def fit(self, X, y):

        n, m = X.shape

        self.weights = np.random.randn(m)
        self.bias = np.random.randn()

        for _ in range(self.num_iterations):

            y_pred = self.predict(X, predict_class=False)

            reg = get_reg(self.reg_l1, self.reg_l2, self.weights)

            dw = (1 / n) * np.dot(X.T, (y_pred - y)) + reg
            db = (1 / n) * np.sum(y_pred - y)

            self.weights -= self.learning_rate * dw
            self.bias -= self.learning_rate * db

    def predict(self, X, predict_class=True):

        linear_model = np.dot(X, self.weights) + self.bias

        y_pred = 1 / (1 + np.exp(-linear_model))

        if predict_class:
            y_pred = np.array([1 if i >= 0.5 else 0 for i in y_pred])

        return y_pred

# **Load, Split and Standardize**

In [14]:
dataset_r = Dataset("regression")
X_train_r, X_test_r, y_train_r, y_test_r = dataset_r.load_split_data()
X_train_r, X_test_r = standardize(X_train_r, X_test_r)

dataset_b = Dataset("binary classification")
X_train_b, X_test_b, y_train_b, y_test_b = dataset_b.load_split_data()
X_train_b, X_test_b = standardize(X_train_b, X_test_b)

# **Train, Test and Compare**

In [15]:
custom_model = CustomLinearRegression(reg_l1=0.1)
custom_model.fit(X_train_r, y_train_r)
y_pred_custom = custom_model.predict(X_test_r)
print(f"Custom Linear Regression L1 (Lasso) MSE: {mean_squared_error(y_test_r, y_pred_custom):.3f}")

custom_model = CustomLinearRegression(reg_l2=0.1)
custom_model.fit(X_train_r, y_train_r)
y_pred_custom =  custom_model.predict(X_test_r)
print(f"Custom Linear Regression L2 (Ridge) MSE: {mean_squared_error(y_test_r, y_pred_custom):.3f}")

custom_model = CustomLinearRegression(reg_l1=0.05, reg_l2=0.05)
custom_model.fit(X_train_r, y_train_r)
y_pred_custom = custom_model.predict(X_test_r)
print(f"Custom Linear Regression Elastic Net MSE: {mean_squared_error(y_test_r, y_pred_custom):.3f}")

Custom Linear Regression L1 (Lasso) MSE: 0.551
Custom Linear Regression L2 (Ridge) MSE: 0.501
Custom Linear Regression Elastic Net MSE: 0.539


In [16]:
sklearn_model = Lasso(alpha=0.1)
sklearn_model.fit(X_train_r, y_train_r)
y_pred_sklearn = sklearn_model.predict(X_test_r)
print(f"Scikit-learn Lasso MSE: {mean_squared_error(y_test_r, y_pred_sklearn):.3f}")

sklearn_model = Ridge(alpha=0.1)
sklearn_model.fit(X_train_r, y_train_r)
y_pred_sklearn = sklearn_model.predict(X_test_r)
print(f"Scikit-learn Ridge MSE: {mean_squared_error(y_test_r, y_pred_sklearn):.3f}")

sklearn_model = ElasticNet(alpha=0.1, l1_ratio=0.5)
sklearn_model.fit(X_train_r, y_train_r)
y_pred_sklearn = sklearn_model.predict(X_test_r)
print(f"Scikit-learn ElasticNet MSE: {mean_squared_error(y_test_r, y_pred_sklearn):.3f}")

Scikit-learn Lasso MSE: 0.664
Scikit-learn Ridge MSE: 0.487
Scikit-learn ElasticNet MSE: 0.611


In [17]:
custom_model = CustomLogisticRegression(reg_l1=0.1)
custom_model.fit(X_train_b, y_train_b)
y_pred_custom = custom_model.predict(X_test_b)
print(f"Custom Logistic Regression L1 Accuracy: {accuracy_score(y_test_b, y_pred_custom):.3f}")

custom_model = CustomLogisticRegression(reg_l2=0.1)
custom_model.fit(X_train_b, y_train_b)
y_pred_custom = custom_model.predict(X_test_b)
print(f"Custom Logistic Regression L2 Accuracy: {accuracy_score(y_test_b, y_pred_custom):.3f}")

custom_model = CustomLogisticRegression(reg_l1=0.05, reg_l2=0.05)
custom_model.fit(X_train_b, y_train_b)
y_pred_custom = custom_model.predict(X_test_b)
print(f"Custom Logistic Regression Elastic Net Accuracy: {accuracy_score(y_test_b, y_pred_custom):.3f}")

Custom Logistic Regression L1 Accuracy: 0.927
Custom Logistic Regression L2 Accuracy: 0.985
Custom Logistic Regression Elastic Net Accuracy: 0.978


In [18]:
sklearn_model = LogisticRegression(penalty="l1", C=1/0.1, solver="saga")
sklearn_model.fit(X_train_b, y_train_b)
y_pred_sklearn = sklearn_model.predict(X_test_b)
print(f"Scikit-learn Logistic Regression L1 Accuracy: {accuracy_score(y_test_b, y_pred_sklearn):.3f}")

sklearn_model = LogisticRegression(penalty="l2", C=1/0.1, solver="lbfgs")
sklearn_model.fit(X_train_b, y_train_b)
y_pred_sklearn = sklearn_model.predict(X_test_b)
print(f"Scikit-learn Logistic Regression L2 Accuracy: {accuracy_score(y_test_b, y_pred_sklearn):.3f}")

sklearn_model = LogisticRegression(penalty="elasticnet", C=1/0.1, l1_ratio=0.5, solver="saga")
sklearn_model.fit(X_train_b, y_train_b)
y_pred_sklearn = sklearn_model.predict(X_test_b)
print(f"Scikit-learn Logistic Regression Elastic Net Accuracy: {accuracy_score(y_test_b, y_pred_sklearn):.3f}")

Scikit-learn Logistic Regression L1 Accuracy: 0.985
Scikit-learn Logistic Regression L2 Accuracy: 0.985
Scikit-learn Logistic Regression Elastic Net Accuracy: 0.985
